# 2. Multi-view aggregation and evaluation

Two things here:

1. **Multi-view** -- why a single heading does not describe a location, and what
   the aggregate reports instead.
2. **Evaluation** -- the three independent levels, run end to end on a small
   worked example.

Still no API key required: the sample set contains a four-heading sweep of one
location, which stands in for what `--multi-view` fetches from Street View.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "samples").exists():
    ROOT = ROOT.parent
SAMPLES = ROOT / "samples" / "images"

BACKEND = "oneformer"
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

from urban_canopy.core.config import CanopyConfig
from urban_canopy.core.pipeline import CanopyPipeline
from urban_canopy.models.factory import build_segmenter

pipeline = CanopyPipeline(segmenter=build_segmenter(BACKEND, device=DEVICE), config=CanopyConfig())
print("ready:", DEVICE)

## One location, four headings

These four frames are the same point, looking north, east, south and west.

In [ ]:
headings = [0, 90, 180, 270]
paths = [SAMPLES / f"streetview_id1_heading{h}.jpg" for h in headings]

results = [pipeline.analyse_image(p) for p in paths]

for h, r in zip(headings, results):
    print(f"heading {h:3d}deg : {r.coverage.tree_coverage_pct:6.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, h, r in zip(axes, headings, results):
    over = r.rgb_image.copy()
    m = r.refined_mask.astype(bool)
    over[m] = (0.45 * np.array([0, 200, 0]) + 0.55 * over[m]).astype("uint8")
    ax.imshow(over)
    ax.set_title(f"{h}deg - {r.coverage.tree_coverage_pct:.1f}%")
    ax.axis("off")
plt.tight_layout()
plt.show()

The spread across one location is the whole point. Reporting whichever heading
you happened to capture would be reporting an arbitrary choice, which is also
why heading selection in this project is deterministic and never driven by the
segmentation output: choosing the view by how well the model segments it would
bias the result upward by construction.

In [ ]:
from urban_canopy.processing.aggregate import aggregate_views

agg = aggregate_views(results)
stats = agg.tree_coverage

print(f"views          {stats.n_valid_views} valid of {stats.n_views}")
print(f"mean           {100 * stats.mean:.2f}%")
print(f"median         {100 * stats.median:.2f}%")
print(f"p25 / p75      {100 * stats.p25:.2f}% / {100 * stats.p75:.2f}%")
print(f"IQR            {100 * stats.iqr:.2f} pp")
print(f"min / max      {100 * stats.minimum:.2f}% / {100 * stats.maximum:.2f}%")

In [ ]:
values = [100 * r.coverage.tree_coverage_ratio for r in results]

plt.figure(figsize=(7, 4))
plt.bar([str(h) for h in headings], values, color="#3f8f4f")
plt.axhline(100 * stats.median, color="crimson", ls="--", label=f"median {100*stats.median:.1f}%")
plt.fill_between([-0.5, 3.5], 100 * stats.p25, 100 * stats.p75,
                 color="crimson", alpha=0.12, label="IQR")
plt.xlim(-0.5, 3.5)
plt.xlabel("heading (degrees)")
plt.ylabel("tree coverage (%)")
plt.legend()
plt.tight_layout()
plt.show()

The median and IQR are reported rather than the mean alone because one heading
pointed at a park and another at a wall is a *skewed* sample, not a noisy one.

Note what the aggregate does **not** contain: a total tree count. Counts stay
per view, because the same tree appears in several headings and no cross-view
association is implemented. Summing them would invent trees.

In [ ]:
print("instance counts per view:", agg.instance_counts)
for note in agg.notes:
    print("note:", note)
print("\nkeys in the aggregate:", sorted(agg.to_dict()))

---

# Evaluation

Three independent levels, against manually annotated ground truth in COCO
Instance Segmentation format (what Roboflow exports).

> **The ground truth below is synthetic.** It is built by perturbing a
> prediction, purely to show the mechanics end to end. The numbers it produces
> say nothing about model quality -- real evaluation needs real annotations,
> drawn under `docs/annotation_protocol.md`.

In [ ]:
import json, tempfile
from urban_canopy.evaluation.rle import encode_rle

work = Path(tempfile.mkdtemp())
pred = results[0]
pred_mask = pred.refined_mask.astype(bool)
h, w = pred_mask.shape

# A "ground truth" that disagrees with the prediction on purpose: shifted right
# by 8 px, so IoU is imperfect while the total area is nearly unchanged.
gt_mask = np.zeros_like(pred_mask)
gt_mask[:, 8:] = pred_mask[:, :-8]

annotations = {
    "images": [{"id": 1, "file_name": paths[0].name, "width": w, "height": h}],
    "categories": [{"id": 1, "name": "tree"}],
    "annotations": [
        {"id": 1, "image_id": 1, "category_id": 1,
         "segmentation": encode_rle(gt_mask), "iscrowd": 0}
    ],
}
(work / "annotations.json").write_text(json.dumps(annotations), encoding="utf-8")

ratio = float(pred_mask.sum()) / pred.coverage.valid_pixels
predictions = {
    "schema": "urban_canopy/predictions/1",
    "manifest": {"note": "notebook demo"},
    "images": [{
        "file_name": paths[0].name, "height": h, "width": w,
        "tree_coverage_ratio": ratio, "tree_coverage_pct": 100 * ratio,
        "tree_source": pred.coverage.tree_source,
        "valid_pixels": pred.coverage.valid_pixels,
        "total_pixels": pred.coverage.total_pixels,
        "exclude_bottom_px": 0,
        "mask": encode_rle(pred_mask),
        "instances": None, "instance_source": None,
        "quality_flags": [], "backend": pred.backend, "class_space": pred.class_space,
    }],
}
(work / "predictions.json").write_text(json.dumps(predictions), encoding="utf-8")
print("wrote fixtures to", work)

In [ ]:
from urban_canopy.evaluation.runner import evaluate_files

report = evaluate_files(work / "predictions.json", work / "annotations.json").to_dict()

micro = report["semantic"]["micro"]
print("LEVEL 1 - pixels")
print(f"  IoU        {micro['iou']:.4f}")
print(f"  Dice / F1  {micro['dice']:.4f}")
print(f"  precision  {micro['precision']:.4f}")
print(f"  recall     {micro['recall']:.4f}")

cov = report["coverage"]
print("\nLEVEL 3 - the coverage indicator")
print(f"  MAE   {cov['mae_pp']:.2f} pp")
print(f"  RMSE  {cov['rmse_pp']:.2f} pp")
print(f"  bias  {cov['bias_pp']:+.2f} pp")

print("\nLEVEL 2 - instances")
print(" ", report["instances_skipped_reason"] or report["instances"])

Three things worth reading off that output:

- **Level 1 and level 3 disagree, and should.** The shifted mask has mediocre
  IoU yet almost perfect coverage error: it covers the wrong pixels but roughly
  the right *amount*. A pipeline reporting only one of the two would hide half
  of what happened.
- **Level 2 was skipped**, with the reason stated. The backend produces stuff
  masks, so there are no instances to match -- rather than reporting zeros that
  would read like a detection failure.
- Correlation is deliberately not a headline metric. A model predicting exactly
  double the true coverage correlates at 1.0 and is wrong by a factor of two.

## The real workflow

```bash
# 1. analyse, exporting predictions with masks
tree-ai --image samples/images/streetview_id1_heading0.jpg --predictions-json

# 2. check the annotation export before trusting it
tree-ai validate-dataset --annotations annotations.json

# 3. evaluate
tree-ai evaluate --predictions artifacts_out/<run>/predictions.json \
                 --annotations annotations.json --report-json report.json
```

See `docs/evaluation.md` for the metric definitions and the validation/test
split policy, and `docs/annotation_protocol.md` for what counts as a tree.